In [7]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install flask torchvision

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 17.7 MB/s eta 0:00:00

   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   -------- ------------------------------- 1/5 [itsdangerous]
   ------------------------ --------------- 3/5 [flask]
   ------------------------ --------------- 3/5 [flask]
   ------------------------ --------------- 3/5 [flask]
   -------------------------------- ------- 4/5 [torchvision]
   -------------------------------- ------- 4/5 [torchvision]
   -------------------------------- ------- 4/5 [torchvision]
   -------------------------------- ------- 4/5 [torchvision]
   -------------------------------- ------- 4/5 [torchvision]
   --------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [11]:
# tests/test_upload.py
import hashlib, io
from unittest.mock import patch
from PIL import Image
import pytest
from app import app   # adjust to your module name

@pytest.fixture
def client():
    app.config["TESTING"] = True
    return app.test_client()

def sha256(b): 
    return hashlib.sha256(b).hexdigest()

def test_route_receives_same_image_bytes(client):
    with open("tests/fixtures/dogs.jpg", "rb") as f:
        original = f.read()

    received = {}
    real_open = Image.open

    def spy_open(fp, *args, **kwargs):
        received["bytes"] = fp.getvalue()   # fp is the BytesIO from your route
        return real_open(fp, *args, **kwargs)

    with patch("app.Image.open", side_effect=spy_open):
        resp = client.post(
            "/upload-photo",
            data={"file": (io.BytesIO(original), "cat.jpg")},
            content_type="multipart/form-data",
        )

    assert resp.status_code == 200
    assert sha256(received["bytes"]) == sha256(original)

test_route_receives_same_image_bytes()
print("Test passed")

TypeError: test_route_receives_same_image_bytes() missing 1 required positional argument: 'client'